In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("spark-local-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 17:35:45 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/14 17:35:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 17:35:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
from pyspark.sql.functions import current_date, current_timestamp

l = [("X",)]
df = spark.createDataFrame(l).toDF("dummy")

df.select(current_date()).show()

df.select(current_timestamp()).show(truncate=False)

+--------------+
|current_date()|
+--------------+
|    2026-10-14|
+--------------+

+--------------------------+
|current_timestamp()       |
+--------------------------+
|2026-10-14 17:37:38.549093|
+--------------------------+



In [11]:
from pyspark.sql.functions import lit, to_date, to_timestamp

df.select(to_date(lit("20261014"), "yyyyMMdd").alias("to_date")).show()

df.select(to_timestamp(lit("20261014 1739"), "yyyyMMdd HHmm").alias("to_timestamp")).show()

+----------+
|   to_date|
+----------+
|2026-10-14|
+----------+

+-------------------+
|       to_timestamp|
+-------------------+
|2026-10-14 17:39:00|
+-------------------+



In [16]:
dts = [
    ("2026-12-20", "2026-12-20 18:09"),
    ("2026-11-21", "2026-11-21 20:21"),
    ("2026-11-14", "2026-11-14 17:34"),
    ("2026-11-15", "2026-11-15 17:39"),
    ("2026-11-21", "2026-11-21 19:20"),
]

In [14]:
df_dts = spark.createDataFrame(dts, schema="date STRING, time STRING")

df_dts.show(truncate=False)

+----------+----------------+
|date      |time            |
+----------+----------------+
|2026-12-20|2026-12-20 18:09|
|2026-11-21|2026-11-21 20:21|
|2026-11-14|2026-11-14 17:34|
|2026-11-15|2026-11-15 17:39|
|2026-11-21|2026-11-21 19:20|
+----------+----------------+



In [23]:
from pyspark.sql.functions import date_add, date_sub

df_dts.select("date", "time"). \
    withColumn("date_add_1", date_add("date", 10)). \
    withColumn("date_add_time", date_add("time", 10)). \
    withColumn("date_sub_10", date_sub("date", 10)). \
    show(truncate=False)

+----------+----------------+----------+-------------+-----------+
|date      |time            |date_add_1|date_add_time|date_sub_10|
+----------+----------------+----------+-------------+-----------+
|2026-12-20|2026-12-20 18:09|2026-12-30|2026-12-30   |2026-12-10 |
|2026-11-21|2026-11-21 20:21|2026-12-01|2026-12-01   |2026-11-11 |
|2026-11-14|2026-11-14 17:34|2026-11-24|2026-11-24   |2026-11-04 |
|2026-11-15|2026-11-15 17:39|2026-11-25|2026-11-25   |2026-11-05 |
|2026-11-21|2026-11-21 19:20|2026-12-01|2026-12-01   |2026-11-11 |
+----------+----------------+----------+-------------+-----------+



In [27]:
from pyspark.sql.functions import current_date, current_timestamp, datediff

df_dts. \
    withColumn("diff_dates", datediff(current_date(), "date")). \
    withColumn("diff_tims", datediff(current_timestamp(), "time")). \
    show(truncate=False) 

+----------+----------------+----------+---------+
|date      |time            |diff_dates|diff_tims|
+----------+----------------+----------+---------+
|2026-12-20|2026-12-20 18:09|-67       |-67      |
|2026-11-21|2026-11-21 20:21|-38       |-38      |
|2026-11-14|2026-11-14 17:34|-31       |-31      |
|2026-11-15|2026-11-15 17:39|-32       |-32      |
|2026-11-21|2026-11-21 19:20|-38       |-38      |
+----------+----------------+----------+---------+



In [30]:
from pyspark.sql.functions import months_between, add_months, round

df_dts. \
    withColumn("months_between_date", round(months_between(current_date(), "date"))). \
    withColumn("months_between_time", round(months_between(current_timestamp(), "time"))). \
    withColumn("add_months_date", add_months("date", 3)). \
    withColumn("add_months_time", add_months("time", 3)). \
    show(truncate=False)

+----------+----------------+-------------------+-------------------+---------------+---------------+
|date      |time            |months_between_date|months_between_time|add_months_date|add_months_time|
+----------+----------------+-------------------+-------------------+---------------+---------------+
|2026-12-20|2026-12-20 18:09|-2.0               |-2.0               |2027-03-20     |2027-03-20     |
|2026-11-21|2026-11-21 20:21|-1.0               |-1.0               |2027-02-21     |2027-02-21     |
|2026-11-14|2026-11-14 17:34|-1.0               |-1.0               |2027-02-14     |2027-02-14     |
|2026-11-15|2026-11-15 17:39|-1.0               |-1.0               |2027-02-15     |2027-02-15     |
|2026-11-21|2026-11-21 19:20|-1.0               |-1.0               |2027-02-21     |2027-02-21     |
+----------+----------------+-------------------+-------------------+---------------+---------------+



In [35]:
from pyspark.sql.functions import trunc

(
    df_dts
        .withColumn("date_trunc_week", trunc("date", "week"))
        .withColumn("date_trunc_month", trunc("date", "month"))
        .withColumn("date_trunc_year", trunc("date", "year"))
        .withColumn("time_trunc_year", trunc("time", "yy"))
        .show()
)

+----------+----------------+---------------+----------------+---------------+---------------+
|      date|            time|date_trunc_week|date_trunc_month|date_trunc_year|time_trunc_year|
+----------+----------------+---------------+----------------+---------------+---------------+
|2026-12-20|2026-12-20 18:09|     2026-12-14|      2026-12-01|     2026-01-01|     2026-01-01|
|2026-11-21|2026-11-21 20:21|     2026-11-16|      2026-11-01|     2026-01-01|     2026-01-01|
|2026-11-14|2026-11-14 17:34|     2026-11-09|      2026-11-01|     2026-01-01|     2026-01-01|
|2026-11-15|2026-11-15 17:39|     2026-11-09|      2026-11-01|     2026-01-01|     2026-01-01|
|2026-11-21|2026-11-21 19:20|     2026-11-16|      2026-11-01|     2026-01-01|     2026-01-01|
+----------+----------------+---------------+----------------+---------------+---------------+



In [37]:
from pyspark.sql.functions import date_trunc

(
    df_dts
        .withColumn("date_dt", date_trunc("HOUR", "date"))
        .withColumn("time_dt", date_trunc("HOUR", "time"))
        .withColumn("time_dti", date_trunc("dd", "time"))
        .show()
)

+----------+----------------+-------------------+-------------------+-------------------+
|      date|            time|            date_dt|            time_dt|           time_dti|
+----------+----------------+-------------------+-------------------+-------------------+
|2026-12-20|2026-12-20 18:09|2026-12-20 00:00:00|2026-12-20 18:00:00|2026-12-20 00:00:00|
|2026-11-21|2026-11-21 20:21|2026-11-21 00:00:00|2026-11-21 20:00:00|2026-11-21 00:00:00|
|2026-11-14|2026-11-14 17:34|2026-11-14 00:00:00|2026-11-14 17:00:00|2026-11-14 00:00:00|
|2026-11-15|2026-11-15 17:39|2026-11-15 00:00:00|2026-11-15 17:00:00|2026-11-15 00:00:00|
|2026-11-21|2026-11-21 19:20|2026-11-21 00:00:00|2026-11-21 19:00:00|2026-11-21 00:00:00|
+----------+----------------+-------------------+-------------------+-------------------+



In [ ]:
from pyspark.sql.functions import dayofyear, month, weekofyear, year

(
    df_dts
        .withColumn("year_from_date", year("date"))
        .withColumn("month_from_date", month("date"))
        .withColumn("weekofyear_from_date", weekofyear("date"))
        .withColumn("dayofyear_from_date", dayofyear("date"))
        .show(truncate=False)
)

+----------+----------------+--------------+---------------+--------------------+-------------------+
|date      |time            |year_from_date|month_from_date|weekofyear_from_date|dayofyear_from_date|
+----------+----------------+--------------+---------------+--------------------+-------------------+
|2026-12-20|2026-12-20 18:09|2026          |12             |51                  |354                |
|2026-11-21|2026-11-21 20:21|2026          |11             |47                  |325                |
|2026-11-14|2026-11-14 17:34|2026          |11             |46                  |318                |
|2026-11-15|2026-11-15 17:39|2026          |11             |46                  |319                |
|2026-11-21|2026-11-21 19:20|2026          |11             |47                  |325                |
+----------+----------------+--------------+---------------+--------------------+-------------------+



In [45]:
from pyspark.sql.functions import current_timestamp, hour, minute, second

(
    df_dts
        .select()
        .withColumn("current_timestamp", current_timestamp())
        .withColumn("hour_from_current_timestamp", hour(current_timestamp()))
        .withColumn("minutes_from_current_timestamp", minute(current_timestamp()))
        .show(truncate=False)
)

+--------------------------+---------------------------+------------------------------+
|current_timestamp         |hour_from_current_timestamp|minutes_from_current_timestamp|
+--------------------------+---------------------------+------------------------------+
|2026-10-14 18:03:50.090686|18                         |3                             |
|2026-10-14 18:03:50.090686|18                         |3                             |
|2026-10-14 18:03:50.090686|18                         |3                             |
|2026-10-14 18:03:50.090686|18                         |3                             |
|2026-10-14 18:03:50.090686|18                         |3                             |
+--------------------------+---------------------------+------------------------------+



In [46]:
spark.stop()